In [2]:
pip install pandas numpy openpyxl


  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.4-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)
Using cached numpy-2.2.4-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


# Procesamiento de Datos en Python

In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from pandas.tseries.holiday import AbstractHolidayCalendar, Holiday
from pandas.tseries.offsets import CustomBusinessDay

# Leer archivos

In [13]:
# Leer archivos desde rutas locales ()
escaneo_df = pd.read_excel(r"D:\prueba_tecnica_deloite\Escaneo_Prueba.xlsx")
grupo_df = pd.read_excel(r"D:\prueba_tecnica_deloite\Grupo_Vulnerabilidades.xlsx")


In [14]:
escaneo_df.head()


,IP,First Detected,Last Detected,CVE ID,Gid,Categoria,Riesgo,Criticidad
0,192168100101,11/30/2022 11:13:23,44816.76875,CVE-2022-4135,1,Actualizacion navegadores,Alto,Transaccional
1,192168100101,44876.917361,44816.76875,"CVE-2022-3887, CVE-2022-3885, CVE-2022-3888, C...",1,Actualizacion navegadores,Bajo,General
2,192168100101,44754.55,44816.76875,"CVE-2022-4262, CVE-2022-44708, CVE-2022-4195, ...",1,Actualizacion navegadores,Alto,General
3,192168100101,07/21/2022 09:54:29,44816.76875,"CVE-2022-34169, CVE-2022-21541, CVE-2022-21540...",2,Actualizacion java,Medio,General
4,192168100101,07/16/2022 21:19:12,07/28/2022 10:24:01,"CVE-2022-34219, CVE-2022-34232, CVE-2022-34229...",3,Actualizacion software,Bajo,General


# Estandarizar IP

In [15]:
# Asegurarse de que cada valor en la columna IP tenga un formato
# válido tipo x.x.x.x, donde cada x es un número entre 0 y 255.
# Convertir IP numérica a x.x.x.x si es válida
def estandarizar_ip(ip):
    try:
        s = str(int(float(ip))).zfill(12)  # Aceptar float también
        if len(s) != 12:
            return np.nan
        partes = [int(s[i:i+3]) for i in range(0, 12, 3)]
        if all(0 <= x <= 255 for x in partes):
            return '.'.join(str(x) for x in partes)
        return np.nan
    except:
        return np.nan


# Aplicar transformación
escaneo_df['IP'] = escaneo_df['IP'].apply(estandarizar_ip)

In [18]:
escaneo_df.head(5)

,IP,First Detected,Last Detected,CVE ID,Gid,Categoria,Riesgo,Criticidad
0,192.168.100.101,11/30/2022 11:13:23,44816.76875,CVE-2022-4135,1,Actualizacion navegadores,Alto,Transaccional
1,192.168.100.101,44876.917361,44816.76875,"CVE-2022-3887, CVE-2022-3885, CVE-2022-3888, C...",1,Actualizacion navegadores,Bajo,General
2,192.168.100.101,44754.55,44816.76875,"CVE-2022-4262, CVE-2022-44708, CVE-2022-4195, ...",1,Actualizacion navegadores,Alto,General
3,192.168.100.101,07/21/2022 09:54:29,44816.76875,"CVE-2022-34169, CVE-2022-21541, CVE-2022-21540...",2,Actualizacion java,Medio,General
4,192.168.100.101,07/16/2022 21:19:12,07/28/2022 10:24:01,"CVE-2022-34219, CVE-2022-34232, CVE-2022-34229...",3,Actualizacion software,Bajo,General


# Merge por Gid

In [19]:
# Hacer merge para traer la columna 'Grupo' desde grupo_df a escaneo_df por Gid
escaneo_df = escaneo_df.merge(
    grupo_df[['Gid', 'Grupo']],  # columnas necesarias
    on='Gid',                    # campo común
    how='left'                   # mantener todos los datos de escaneo_df
)


In [20]:
escaneo_df.head()

,IP,First Detected,Last Detected,CVE ID,Gid,Categoria,Riesgo,Criticidad,Grupo
0,192.168.100.101,11/30/2022 11:13:23,44816.76875,CVE-2022-4135,1,Actualizacion navegadores,Alto,Transaccional,Actualizacion Microsoft Edge
1,192.168.100.101,44876.917361,44816.76875,"CVE-2022-3887, CVE-2022-3885, CVE-2022-3888, C...",1,Actualizacion navegadores,Bajo,General,Actualizacion Microsoft Edge
2,192.168.100.101,44754.55,44816.76875,"CVE-2022-4262, CVE-2022-44708, CVE-2022-4195, ...",1,Actualizacion navegadores,Alto,General,Actualizacion Microsoft Edge
3,192.168.100.101,07/21/2022 09:54:29,44816.76875,"CVE-2022-34169, CVE-2022-21541, CVE-2022-21540...",2,Actualizacion java,Medio,General,Actualizacion Oracle Java SE
4,192.168.100.101,07/16/2022 21:19:12,07/28/2022 10:24:01,"CVE-2022-34219, CVE-2022-34232, CVE-2022-34229...",3,Actualizacion software,Bajo,General,Actualizacion Adobe Acrobat y Reader


# Agrupación , ordenar por riegos y crear archivo Escaneo_Agrupado.xlsx

In [ ]:
# Definir prioridad del riesgo
riesgo_prioridad = {'Alto': 0, 'Medio': 1, 'Bajo': 2}
escaneo_df['Riesgo_Orden'] = escaneo_df['Riesgo'].map(riesgo_prioridad)

# Agrupar y tomar la primera fila por grupo 
agrupado_df = escaneo_df.groupby(['IP', 'Categoria', 'Riesgo'], as_index=False).first()

# Ordenar por IP y Riesgo de Alto a Bajo
agrupado_df = agrupado_df.sort_values(by=['IP', 'Riesgo_Orden'])

# Quitar columna auxiliar
agrupado_df = agrupado_df.drop(columns='Riesgo_Orden')

# Guardar resultado
agrupado_df.to_excel(r"D:\prueba_tecnica_deloite\Primer_punto\Escaneo_Agrupado.xlsx", index=False)


# punto 3 crear calendario Leer el archivo agrupado


In [22]:
# 1. Leer el archivo agrupado
df = pd.read_excel(r"D:\prueba_tecnica_deloite\Primer_punto\Escaneo_Agrupado.xlsx")

In [23]:
df.head()

,IP,Categoria,Riesgo,First Detected,Last Detected,CVE ID,Gid,Criticidad,Grupo
0,192.168.100.100,Actualizacion java,Alto,45451.725,45750.452778,"CVE-2023-41993, CVE-2024-21085, CVE-2024-21011...",2,General,Actualizacion Oracle Java SE
1,192.168.100.100,Actualizacion navegadores,Alto,45963.588889,45750.452778,CVE-2024-9680,12,General,Actualizacion Mozilla Firefox
2,192.168.100.100,Actualizacion windows,Alto,45963.588889,45993.846528,"CVE-2025-21217, CVE-2025-21246, CVE-2025-21334...",5,General,Actualizacion Microsoft Windows
3,192.168.100.100,Actualizacion java,Medio,45963.588889,45750.452778,"CVE-2023-42950, CVE-2024-25062, CVE-2024-21235...",2,General,Actualizacion Oracle Java SE
4,192.168.100.100,Actualizacion navegadores,Medio,45963.588889,45750.452778,"CVE-2024-9395, CVE-2024-9397, CVE-2024-9392, C...",12,General,Actualizacion Mozilla Firefox


#  Ordenar por Riesgo (prioridad)

In [24]:
prioridad_riesgo = {'Alto': 0, 'Medio': 1, 'Bajo': 2}
df['Riesgo_Orden'] = df['Riesgo'].map(prioridad_riesgo)
df = df.sort_values(by='Riesgo_Orden').reset_index(drop=True)

# Ordenar por Riesgo (prioridad)
prioridad_riesgo = {'Alto': 0, 'Medio': 1, 'Bajo': 2}
df['Riesgo_Orden'] = df['Riesgo'].map(prioridad_riesgo)
df = df.sort_values(by='Riesgo_Orden').reset_index(drop=True)

#  Definir calendario hábil (lunes a viernes excluyendo festivos)
class FestivosColombia(AbstractHolidayCalendar):
    rules = [
        Holiday('Jueves Santo', year=2025, month=4, day=17),
        Holiday('Viernes Santo', year=2025, month=4, day=18),
    ]

festivos = FestivosColombia().holidays(start='2025-04-01', end='2025-04-30')
business_days = CustomBusinessDay(holidays=festivos)

#  Generar fechas hábiles del mes de abril
fecha_inicio = pd.Timestamp('2025-04-01')
fechas_disponibles = pd.date_range(start=fecha_inicio, end='2025-04-30', freq=business_days)

#  Asignar 5 actividades por día
total = len(df)
fechas_asignadas = []

i = 0
for fecha in fechas_disponibles:
    for _ in range(5):
        if i >= total:
            break
        fechas_asignadas.append(fecha)
        i += 1
    if i >= total:
        break

# Rellenar columna
df['Fecha_Remediacion'] = fechas_asignadas + [np.nan] * (total - len(fechas_asignadas))

# 6. Eliminar columna auxiliar y guardar
df = df.drop(columns='Riesgo_Orden')
df.to_excel(r"D:\prueba_tecnica_deloite\Primer_punto\Calendario_Remediacion_Abril2025.xlsx", index=False)